# Embed the last missing English row (id=2000) into Qdrant

`knowledge_base/Womens_Health_KB_English - 2000_final.csv` just went from 1999 to 2000 rows -- `id=2000` (the backfilled Sindhi cycle-day-1 row) finally has a translation. Everything else is already embedded (3999/4000 points in `naari_ai_kb`). Rather than re-running the full `embed_english_kb.ipynb` batch job for one row, this embeds and upserts just this one point at `point_id = 10000 + 2000 = 12000`, matching that notebook's ID scheme and payload shape exactly.

**Runtime -> Change runtime type -> T4 GPU** (or CPU is fine for a single row, just slower to load the model).

In [ ]:
!pip install -q qdrant-client FlagEmbedding

In [ ]:
!rm -rf naari-ai
!git clone --branch sana/test-protection --depth 1 https://github.com/sana200420/naari-ai.git
%cd naari-ai

import sys
sys.path.insert(0, ".")

from retrieval.normalize import normalize_sd

print("cloned + imported OK")

In [ ]:
import csv

with open("knowledge_base/Womens_Health_KB_English - 2000_final.csv", encoding="utf-8", newline="") as f:
    en_rows = list(csv.DictReader(f))

print(f"{len(en_rows)} English rows loaded (expect 2000)")
assert len(en_rows) == 2000, f"got {len(en_rows)}, not 2000 -- check the CSV made it into this clone"

row = next(r for r in en_rows if r["id"] == "2000")
print("row 2000:")
for k, v in row.items():
    print(f"  {k}: {v}")

In [ ]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
print("model loaded")

In [ ]:
from getpass import getpass
from qdrant_client import QdrantClient

QDRANT_URL = getpass("Qdrant cluster URL: ")
QDRANT_API_KEY = getpass("Qdrant API key: ")

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

before_count = client.get_collection("naari_ai_kb").points_count
print(f"collection currently holds {before_count} points (expect 3999)")

In [ ]:
from qdrant_client import models

# Same convention as embed_and_index.ipynb / embed_english_kb.ipynb: embed the
# question, not the answer -- retrieval matches a user's query against KB
# questions, the answer is only ever returned from the payload.
normalized_question = normalize_sd(row["question"])
out = model.encode([normalized_question], return_dense=True, return_sparse=True, return_colbert_vecs=False)
dense_vec = out["dense_vecs"][0]
sparse_weights = out["lexical_weights"][0]

sparse_vector = models.SparseVector(
    indices=[int(k) for k in sparse_weights.keys()],
    values=[float(v) for v in sparse_weights.values()],
)

ENGLISH_POINT_ID_OFFSET = 10000
answer_id = int(row["id"])
point_id = ENGLISH_POINT_ID_OFFSET + answer_id

point = models.PointStruct(
    id=point_id,
    vector={
        "dense": dense_vec.tolist(),
        "sparse": sparse_vector,
    },
    payload={
        "answer_id": answer_id,
        "category": row["category"],
        "sub_category": row["sub_category"],
        "question": row["question"],
        "answer": row["answer"],
        "source": row["source"],
        "review_tier": row["review_tier"],
        "lang": "en",
    },
)

print(f"upserting point_id={point_id} (answer_id={answer_id})")
client.upsert(collection_name="naari_ai_kb", points=[point])

after_count = client.get_collection("naari_ai_kb").points_count
print(f"collection now holds {after_count} points (expect {before_count + 1} = 4000)")
assert after_count == before_count + 1

In [ ]:
# Verify by reading the point straight back from Qdrant, not just trusting the count.
fetched = client.retrieve(collection_name="naari_ai_kb", ids=[12000], with_payload=True, with_vectors=False)
assert len(fetched) == 1, "point 12000 not found after upsert"
p = fetched[0]
print("payload:", p.payload)
assert p.payload["answer_id"] == 2000
assert p.payload["lang"] == "en"
assert p.payload["question"] == row["question"]
print("\nverified: point 12000 matches the expected row. Collection is now 4000/4000.")